# OddVision — Passo 2: exploração guiada dos dados reais

**Objetivo:** entender o JSON do StatsBomb Open Data (Copa do Mundo 2022) **antes** de escrever o pipeline.

**Regras deste notebook:**
- **Python puro:** somente `json`, `collections`, `urllib` e `math`. Não usa pandas nem código do pacote `oddvision`. Por isso, ele serve de **oráculo independente** para os testes do pipeline.
- Os arquivos são lidos **em memória**, a partir de URLs fixadas em um commit. **Nada é gravado em disco.** A ingestão oficial é o Passo 3.
- Cada seção responde a uma pergunta da spec, ou confirma um dos **7 caminhos ⚠️** de `docs/data_dictionary.md`.

| Seção | Pergunta |
|---|---|
| 1 | Configuração e commit fixado |
| 2 | `competitions.json` e `matches` |
| 3 | Carga das 64 partidas (events + lineups) |
| 4 | Perfil dos eventos (granularidade, chaves, nulos, domínios) |
| 5 | Escalações e ⚠️ critério de titular |
| 6 | ⚠️ `shot.end_location` (altura `end_z`) |
| 7 | ⚠️ passe em profundidade |
| 8 | ⚠️ cartões (`foul_committed.card`, `bad_behaviour.card`) |
| 9 | ⚠️ substituições (`substitution.replacement`, `substitution.outcome`) |
| 10 | `Half End` e duração dos períodos (necessário para a M8) |
| 11 | Integridade entre arquivos (placar, times, jogadores) |
| 12 | Partidas-sentinela |
| 13 | Valores de referência da final (M1–M8) |

## 1. Configuração e commit fixado

O repositório `statsbomb/open-data` foi transferido para **`hudl/open-data`**. O nome antigo ainda redireciona.

Fixamos o **commit** `4b73468f…` (07/09/2026, o topo do `master` em 29/09/2026). Isso garante que os mesmos arquivos sejam lidos sempre (D04).

In [1]:
import json
import re
import sys
import urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from math import hypot

SOURCE_REPO = "hudl/open-data"  # antigo statsbomb/open-data (redireciona)
SOURCE_COMMIT = "4b73468fc5b0f1950f9f66fada70ad3a4f9327cb"  # 2026-09-07
BASE_URL = f"https://raw.githubusercontent.com/{SOURCE_REPO}/{SOURCE_COMMIT}/data/"
COMPETITION_ID, SEASON_ID = 43, 106
FINAL_ID = 3869685


def fetch_json(path):
    """Lê um arquivo da fonte fixada, em memória. Nada é gravado em disco."""
    with urllib.request.urlopen(BASE_URL + path, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))


def show(counter, n=None):
    for key, value in counter.most_common(n):
        print(f"{value:>8}  {key}")


def clock_to_seconds(text):
    """'MM:SS' (lineups) -> segundos."""
    minutes, seconds = text.split(":")
    return int(minutes) * 60 + int(seconds)


def timestamp_to_seconds(text):
    """'HH:MM:SS.mmm' (events) -> segundos."""
    hours, minutes, seconds = text.split(":")
    return int(hours) * 3600 + int(minutes) * 60 + float(seconds)


print("Python", sys.version.split()[0])
print("Fonte:", BASE_URL)

Python 3.13.7
Fonte: https://raw.githubusercontent.com/hudl/open-data/4b73468fc5b0f1950f9f66fada70ad3a4f9327cb/data/


## 2. `competitions.json` e arquivo de partidas

**Granularidade esperada:**
- `competitions`: um par (competição, temporada) por item.
- `matches`: uma partida por item.

In [2]:
competitions = fetch_json("competitions.json")
print("pares competição/temporada no arquivo:", len(competitions))
print("chaves:", sorted(competitions[0]))
wc = [c for c in competitions if (c["competition_id"], c["season_id"]) == (COMPETITION_ID, SEASON_ID)]
assert len(wc) == 1
print(json.dumps(wc[0], indent=1, ensure_ascii=False))

pares competição/temporada no arquivo: 80
chaves: ['competition_gender', 'competition_id', 'competition_international', 'competition_name', 'competition_youth', 'country_name', 'match_available', 'match_available_360', 'match_updated', 'match_updated_360', 'season_id', 'season_name']
{
 "competition_id": 43,
 "season_id": 106,
 "country_name": "International",
 "competition_name": "FIFA World Cup",
 "competition_gender": "male",
 "competition_youth": false,
 "competition_international": true,
 "season_name": "2022",
 "match_updated": "2026-05-04T01:48:57.914346",
 "match_updated_360": "2026-05-04T01:53:40.309717",
 "match_available_360": "2026-05-04T01:53:40.309717",
 "match_available": "2026-05-04T01:48:57.914346"
}


In [3]:
matches = fetch_json(f"matches/{COMPETITION_ID}/{SEASON_ID}.json")
match_ids = [m["match_id"] for m in matches]
print("partidas:", len(matches), "| match_id únicos:", len(set(match_ids)) == len(match_ids))

teams = {}
for m in matches:
    for side in ("home", "away"):
        t = m[f"{side}_team"]
        teams[t[f"{side}_team_id"]] = t[f"{side}_team_name"]
print("seleções distintas:", len(teams))
print("chaves de uma partida:", sorted(matches[0]))

presence = Counter(k for m in matches for k in m)
print("chaves ausentes em alguma partida:", {k: len(matches) - v for k, v in presence.items() if v < len(matches)} or "nenhuma")
print("valores nulos por chave:", dict(Counter(k for m in matches for k, v in m.items() if v is None)) or "nenhum")
print("fases:", dict(Counter(m["competition_stage"]["name"] for m in matches)))
print("versões (metadata):", Counter(json.dumps(m["metadata"], sort_keys=True) for m in matches).most_common())
print("formato de kick_off:", Counter(bool(re.fullmatch(r"\d{2}:\d{2}:\d{2}\.\d{3}", m["kick_off"] or "")) for m in matches))

final = next(m for m in matches if m["match_id"] == FINAL_ID)
print("\nfinal:", final["match_date"], "|", final["home_team"]["home_team_name"], final["home_score"], "x",
      final["away_score"], final["away_team"]["away_team_name"], "|", final["competition_stage"]["name"])

partidas: 64 | match_id únicos: True
seleções distintas: 32
chaves de uma partida: ['away_score', 'away_team', 'competition', 'competition_stage', 'home_score', 'home_team', 'kick_off', 'last_updated', 'last_updated_360', 'match_date', 'match_id', 'match_status', 'match_status_360', 'match_week', 'metadata', 'referee', 'season', 'stadium']
chaves ausentes em alguma partida: {'referee': 1}
valores nulos por chave: nenhum
fases: {'Group Stage': 48, 'Quarter-finals': 4, 'Final': 1, '3rd Place Final': 1, 'Round of 16': 8, 'Semi-finals': 2}
versões (metadata): [('{"data_version": "1.1.0", "shot_fidelity_version": "2", "xy_fidelity_version": "2"}', 64)]
formato de kick_off: Counter({True: 64})

final: 2022-12-18 | Argentina 3 x 3 France | Final


## 3. Carga das 64 partidas (em memória)

Leitura de 64 arquivos de eventos (cerca de 3,7 MB cada) e 64 de escalações. Leva de 1 a 2 minutos.

In [4]:
def load_match_files(match_id):
    return match_id, fetch_json(f"events/{match_id}.json"), fetch_json(f"lineups/{match_id}.json")


with ThreadPoolExecutor(max_workers=8) as pool:
    loaded = list(pool.map(load_match_files, match_ids))

events = {mid: ev for mid, ev, _ in loaded}
lineups = {mid: lu for mid, _, lu in loaded}
all_events = [(mid, e) for mid, ev in events.items() for e in ev]
print("arquivos de events:", len(events), "| arquivos de lineups:", len(lineups))
print("total de arquivos da ingestão (1 competitions + 1 matches + events + lineups):", 2 + len(events) + len(lineups))
print("total de eventos:", len(all_events))
print("eventos na final:", len(events[FINAL_ID]))

arquivos de events: 64 | arquivos de lineups: 64
total de arquivos da ingestão (1 competitions + 1 matches + events + lineups): 130
total de eventos: 234637
eventos na final: 4407


## 4. Perfil dos eventos

Perguntas da skill `data:explore-data` aplicadas ao núcleo `events`:
- **Granularidade:** um evento.
- **Chave:** `id`. É única?
- Qual a taxa de presença de cada chave de primeiro nível?
- Quais são os domínios dos campos categóricos?
- Há valores suspeitos?

In [5]:
n = len(all_events)
presence = Counter(k for _, e in all_events for k in e)
print(f"{'chave':<22}{'presença':>10}")
for key, value in presence.most_common():
    print(f"{key:<22}{value / n:>10.1%}")

ids = [e["id"] for _, e in all_events]
print("\nids únicos (entre as 64 partidas):", len(set(ids)) == n)
print("'index' = 1..N, em ordem, em todas as partidas:",
      all([e["index"] for e in ev] == list(range(1, len(ev) + 1)) for ev in events.values()))
print("chave 'match_id' dentro de algum evento:", presence.get("match_id", 0))
print("timestamps no formato HH:MM:SS.mmm:",
      all(re.fullmatch(r"\d{2}:\d{2}:\d{2}\.\d{3}", e["timestamp"]) for _, e in all_events))

chave                   presença
id                        100.0%
index                     100.0%
period                    100.0%
timestamp                 100.0%
minute                    100.0%
second                    100.0%
type                      100.0%
possession                100.0%
possession_team           100.0%
play_pattern              100.0%
team                      100.0%
player                     99.5%
position                   99.5%
location                   99.1%
related_events             96.0%
duration                   72.9%
pass                       29.2%
carry                      22.9%
under_pressure             16.5%
ball_receipt                3.1%
counterpress                2.6%
duel                        1.9%
off_camera                  1.4%
clearance                   1.1%
out                         0.8%
dribble                     0.8%
goalkeeper                  0.8%
shot                        0.6%
interception                0.6%
foul_won  

In [6]:
types = Counter(e["type"]["name"] for _, e in all_events)
print("tipos de evento:", len(types))
show(types)

tipos de evento: 33
   68515  Pass
   63699  Ball Receipt*
   53764  Carry
   16554  Pressure
    5821  Ball Recovery
    4389  Duel
    2684  Clearance
    2386  Block
    1793  Dribble
    1790  Goal Keeper
    1775  Foul Committed
    1755  Miscontrol
    1693  Foul Won
    1494  Shot
    1431  Dispossessed
    1371  Interception
    1036  Dribbled Past
     587  Substitution
     403  Injury Stoppage
     286  Half Start
     286  Half End
     243  Tactical Shift
     236  50/50
     162  Referee Ball-Drop
     128  Starting XI
     104  Shield
      74  Player Off
      74  Player On
      44  Bad Behaviour
      28  Error
      26  Offside
       3  Own Goal For
       3  Own Goal Against


In [7]:
print("eventos por período:", dict(sorted(Counter(e["period"] for _, e in all_events).items())))
minute_range = defaultdict(lambda: [10**6, -1])
for _, e in all_events:
    r = minute_range[e["period"]]
    r[0], r[1] = min(r[0], e["minute"]), max(r[1], e["minute"])
print("minuto min/max por período:", dict(sorted(minute_range.items())))

locs = [e["location"] for _, e in all_events if "location" in e]
print("\ntamanho de 'location':", Counter(len(l) for l in locs))
print("x: [", min(l[0] for l in locs), ",", max(l[0] for l in locs), "]  y: [",
      min(l[1] for l in locs), ",", max(l[1] for l in locs), "]")
print("tipos sem 'location':", dict(Counter(e["type"]["name"] for _, e in all_events if "location" not in e)))
print("tipos sem 'player':", dict(Counter(e["type"]["name"] for _, e in all_events if "player" not in e)))
print("tipos sem 'duration':", dict(Counter(e["type"]["name"] for _, e in all_events if "duration" not in e)))
print("valores de under_pressure:", Counter(e.get("under_pressure", "<ausente>") for _, e in all_events))

eventos por período: {1: 116337, 2: 112708, 3: 2885, 4: 2601, 5: 106}
minuto min/max por período: {1: [0, 59], 2: [45, 103], 3: [90, 108], 4: [105, 124], 5: [120, 127]}

tamanho de 'location': Counter({2: 232512})
x: [ 0.1 , 120.0 ]  y: [ 0.1 , 80.0 ]
tipos sem 'location': {'Starting XI': 128, 'Half Start': 286, 'Tactical Shift': 243, 'Half End': 286, 'Injury Stoppage': 403, 'Substitution': 587, 'Player Off': 74, 'Player On': 74, 'Bad Behaviour': 44}
tipos sem 'player': {'Starting XI': 128, 'Half Start': 286, 'Tactical Shift': 243, 'Own Goal For': 3, 'Half End': 286, 'Referee Ball-Drop': 162}
tipos sem 'duration': {'Ball Receipt*': 63699, 'Pass': 1}
valores de under_pressure: Counter({'<ausente>': 195816, True: 38821})


In [8]:
shots = [(mid, e) for mid, e in all_events if e["type"]["name"] == "Shot"]
passes = [(mid, e) for mid, e in all_events if e["type"]["name"] == "Pass"]
carries = [(mid, e) for mid, e in all_events if e["type"]["name"] == "Carry"]

print("shot.outcome:"); show(Counter(e["shot"]["outcome"]["name"] for _, e in shots))
print("shot.type:"); show(Counter(e["shot"]["type"]["name"] for _, e in shots))
print("statsbomb_xg ausente:", sum("statsbomb_xg" not in e["shot"] for _, e in shots),
      "| fora de [0,1]:", sum(not 0 <= e["shot"]["statsbomb_xg"] <= 1 for _, e in shots))
print("xG dos pênaltis:", Counter(round(e["shot"]["statsbomb_xg"], 4) for _, e in shots if e["shot"]["type"]["name"] == "Penalty"))
print("menor x de finalização (períodos 1–4):", min(e["location"][0] for _, e in shots if e["period"] <= 4))

print("\npass.outcome (ausente = completo):"); show(Counter(e["pass"].get("outcome", {}).get("name", "<ausente>") for _, e in passes))
print("pass.type (ausente = passe regular):"); show(Counter(e["pass"].get("type", {}).get("name", "<ausente>") for _, e in passes))
print("passes sem end_location:", sum("end_location" not in e["pass"] for _, e in passes),
      "| completos sem recipient:", sum("outcome" not in e["pass"] and "recipient" not in e["pass"] for _, e in passes))
print("chaves de carry:", Counter(k for _, e in carries for k in e["carry"]))

shot.outcome:
     448  Off T
     379  Blocked
     346  Saved
     195  Goal
      85  Wayward
      36  Post
       3  Saved to Post
       2  Saved Off Target
shot.type:
    1382  Open Play
      64  Penalty
      46  Free Kick
       2  Corner
statsbomb_xg ausente: 0 | fora de [0,1]: 0
xG dos pênaltis: Counter({0.7835: 64})
menor x de finalização (períodos 1–4): 59.0

pass.outcome (ausente = completo):
   56346  <ausente>
   10393  Incomplete
    1116  Out
     394  Unknown
     224  Pass Offside
      42  Injury Clearance
pass.type (ausente = passe regular):
   56423  <ausente>
    5349  Recovery
    2617  Throw-in
    1768  Free Kick
    1002  Goal Kick
     569  Corner
     480  Interception
     307  Kick Off
passes sem end_location: 0 | completos sem recipient: 0
chaves de carry: Counter({'end_location': 53764})


## 5. Escalações e ⚠️ critério de titular

A hipótese do dicionário de dados é: `is_starter = positions[0].start_reason == 'Starting XI'`.

Vamos verificar três coisas:
1. Quais valores `start_reason` aparecem na **primeira** posição de cada jogador.
2. Se `positions` já vem ordenado no tempo.
3. **Validação cruzada:** a regra reproduz o evento `Starting XI`, que é outra fonte dentro dos mesmos dados?

In [9]:
lu = lineups[FINAL_ID]
print("formato do arquivo:", type(lu).__name__, "| times:", [t["team_name"] for t in lu])
print("chaves do time:", sorted(lu[0]))
print("'match_id' dentro do arquivo de lineups:", '"match_id"' in json.dumps(lu))
player = lu[0]["lineup"][0]
print("chaves do jogador:", sorted(player))
print("chaves de uma posição:", sorted(next(p for t in lu for p in t["lineup"] if p["positions"])["positions"][0]))
print("chaves de um cartão (lineups):", sorted(next(c for t in lu for p in t["lineup"] for c in p["cards"])))

formato do arquivo: list | times: ['France', 'Argentina']
chaves do time: ['lineup', 'team_id', 'team_name']
'match_id' dentro do arquivo de lineups: False
chaves do jogador: ['cards', 'country', 'jersey_number', 'player_id', 'player_name', 'player_nickname', 'positions']
chaves de uma posição: ['end_reason', 'from', 'from_period', 'position', 'position_id', 'start_reason', 'to', 'to_period']
chaves de um cartão (lineups): ['card_type', 'period', 'reason', 'time']


In [10]:
first_reason = Counter()
positions_sorted = True
starters = Counter()
not_played = 0
for mid, lu in lineups.items():
    for team in lu:
        for pl in team["lineup"]:
            pos = pl["positions"]
            if not pos:
                not_played += 1
                continue
            first_reason[pos[0]["start_reason"]] += 1
            keys = [(p["from_period"], clock_to_seconds(p["from"])) for p in pos]
            positions_sorted &= keys == sorted(keys)
            if pos[0]["start_reason"] == "Starting XI":
                starters[(mid, team["team_id"])] += 1

print("start_reason da primeira posição:"); show(first_reason)
print("positions já vem em ordem cronológica:", positions_sorted)
print("relacionados que não entraram (positions vazio):", not_played)
print("combinações time×partida:", len(starters), "| titulares por combinação:", Counter(starters.values()))

mismatch = 0
for mid, ev in events.items():
    for e in ev:
        if e["type"]["name"] == "Starting XI":
            xi = {p["player"]["id"] for p in e["tactics"]["lineup"]}
            team = next(t for t in lineups[mid] if t["team_id"] == e["team"]["id"])
            rule = {pl["player_id"] for pl in team["lineup"]
                    if pl["positions"] and pl["positions"][0]["start_reason"] == "Starting XI"}
            mismatch += xi != rule
print("times em que a regra difere do evento 'Starting XI':", mismatch)

start_reason da primeira posição:
    1408  Starting XI
     492  Substitution - On (Tactical)
      56  Tactical Shift
      36  Substitution - On (Injury)
       3  Substitution - On (Off Camera)
positions já vem em ordem cronológica: False
relacionados que não entraram (positions vazio): 1249
combinações time×partida: 128 | titulares por combinação: Counter({11: 128})
times em que a regra difere do evento 'Starting XI': 0


### 5.1 `positions` fora de ordem

A célula acima mostrou dois sinais de alerta:
- `positions` **não** vem sempre em ordem cronológica;
- 56 jogadores têm "Tactical Shift" como primeira posição listada.

Abaixo verificamos o que a ordenação por `(from_period, from)` resolve e se ela afeta a regra de titular.

In [11]:
def sort_positions(positions):
    """Ordenação estável por (período, relógio). Em caso de empate, mantém a ordem da fonte."""
    return sorted(positions, key=lambda p: (p["from_period"], clock_to_seconds(p["from"])))


shift_first = [(mid, t["team_name"], pl) for mid, lu in lineups.items() for t in lu for pl in t["lineup"]
               if pl["positions"] and pl["positions"][0]["start_reason"] == "Tactical Shift"]
print("1ª posição listada = 'Tactical Shift':", len(shift_first))
print("1ª posição após ordenar:", dict(Counter(sort_positions(pl["positions"])[0]["start_reason"] for _, _, pl in shift_first)))

changed = sum((pl["positions"][0]["start_reason"] == "Starting XI") != (sort_positions(pl["positions"])[0]["start_reason"] == "Starting XI")
              for lu in lineups.values() for t in lu for pl in t["lineup"] if pl["positions"])
ties = sum(len({(p["from_period"], clock_to_seconds(p["from"])) for p in pl["positions"]}) < len(pl["positions"])
           for lu in lineups.values() for t in lu for pl in t["lineup"])
print("titulares que mudam com a ordenação:", changed)
print("jogadores com empate em (from_period, from):", ties)

mid, team, pl = next(x for x in shift_first if sort_positions(x[2]["positions"])[0]["start_reason"] == "Tactical Shift")
print(f"\nexemplo que continua 'Tactical Shift' após ordenar: {mid} {team} — {pl['player_name']}")
for p in pl["positions"]:
    print(f"  P{p['from_period']} {p['from']} -> P{p['to_period']} {p['to']} | {p['start_reason']} -> {p['end_reason']} | {p['position']}")

1ª posição listada = 'Tactical Shift': 56
1ª posição após ordenar: {'Tactical Shift': 45, 'Substitution - On (Injury)': 1, 'Substitution - On (Tactical)': 10}
titulares que mudam com a ordenação: 0
jogadores com empate em (from_period, from): 28

exemplo que continua 'Tactical Shift' após ordenar: 3857271 Iran — Saeid Ezatolahi Afagh
  P2 45:00 -> P2 76:43 | Tactical Shift -> Tactical Shift | Left Defensive Midfield
  P2 76:43 -> PNone None | Tactical Shift -> Final Whistle | Center Defensive Midfield


**Leitura dos resultados:**
- A regra de **titular** é robusta: `positions[0].start_reason == 'Starting XI'` coincide com o evento `Starting XI` em todos os times, e a ordenação não muda nenhum titular.
- A regra de **"entrou em campo"** (`positions` não vazio) também não depende da ordem.
- Já a **primeira posição ocupada** (`first_position`) depende da ordem. Por isso o pipeline deve ordenar as posições antes (ordenação estável).
- Alguns reservas têm a entrada registrada como "Tactical Shift", e não como "Substitution - On". Isso é uma **peculiaridade da fonte**. Por isso o OddVision **não** usa `start_reason` para identificar reservas utilizados: usa `played AND NOT is_starter`.

## 6. ⚠️ `shot.end_location` — existe a altura (`end_z`)?

In [12]:
print("tamanho de shot.end_location:", Counter(len(e["shot"]["end_location"]) for _, e in shots))
print("tamanho por resultado:")
show(Counter((e["shot"]["outcome"]["name"], len(e["shot"]["end_location"])) for _, e in shots))

tamanho de shot.end_location: Counter({3: 1030, 2: 464})
tamanho por resultado:
     448  ('Off T', 3)
     379  ('Blocked', 2)
     346  ('Saved', 3)
     195  ('Goal', 3)
      85  ('Wayward', 2)
      36  ('Post', 3)
       3  ('Saved to Post', 3)
       2  ('Saved Off Target', 3)


## 7. ⚠️ Passe em profundidade: `pass.through_ball` ou `pass.technique`?

In [13]:
print("pass.through_ball:", Counter(repr(e["pass"].get("through_ball", "<ausente>")) for _, e in passes))
print("pass.technique:"); show(Counter(e["pass"].get("technique", {}).get("name", "<ausente>") for _, e in passes))
flag = {e["id"] for _, e in passes if e["pass"].get("through_ball")}
tech = {e["id"] for _, e in passes if e["pass"].get("technique", {}).get("name") == "Through Ball"}
print("through_ball=True:", len(flag), "| technique='Through Ball':", len(tech), "| em ambos:", len(flag & tech))

pass.through_ball: Counter({"'<ausente>'": 68261, 'True': 254})
pass.technique:
   67801  <ausente>
     254  Through Ball
     234  Inswinging
     206  Outswinging
      20  Straight
through_ball=True: 254 | technique='Through Ball': 254 | em ambos: 254


## 8. ⚠️ Cartões

Hipótese: os cartões estão em `foul_committed.card` e `bad_behaviour.card`.

Primeiro procuramos a chave `card` em **qualquer** lugar dos eventos, para não deixar passar outro caminho. Depois conciliamos com os cartões registrados nas escalações.

In [14]:
def find_key_paths(obj, target, path="", out=None):
    out = Counter() if out is None else out
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{path}.{k}" if path else k
            if k == target:
                out[p] += 1
            find_key_paths(v, target, p, out)
    elif isinstance(obj, list):
        for v in obj:
            find_key_paths(v, target, path + "[]", out)
    return out


card_paths = Counter()
for _, e in all_events:
    find_key_paths(e, "card", out=card_paths)
print("caminhos da chave 'card' nos eventos:"); show(card_paths)

cards_events = Counter()
for mid, e in all_events:
    for key in ("foul_committed", "bad_behaviour"):
        card = e.get(key, {}).get("card")
        if card:
            cards_events[(mid, e.get("player", {}).get("id"), card["name"])] += 1
print("nomes de cartão:", Counter(name for (_, _, name), c in cards_events.items() for _ in range(c)))

cards_lineups = Counter()
for mid, lu in lineups.items():
    for t in lu:
        for pl in t["lineup"]:
            for c in pl["cards"]:
                cards_lineups[(mid, pl["player_id"], c["card_type"])] += 1
print("total nos eventos:", sum(cards_events.values()), "| total nas escalações:", sum(cards_lineups.values()))
print("só nos eventos:", dict(cards_events - cards_lineups) or "nada")
print("só nas escalações:", dict(cards_lineups - cards_events) or "nada")

caminhos da chave 'card' nos eventos:
     184  foul_committed.card
      44  bad_behaviour.card
nomes de cartão: Counter({'Yellow Card': 224, 'Second Yellow': 3, 'Red Card': 1})
total nos eventos: 228 | total nas escalações: 228
só nos eventos: nada
só nas escalações: nada


## 9. ⚠️ Substituições

In [15]:
subs = [(mid, e) for mid, e in all_events if e["type"]["name"] == "Substitution"]
print("substituições:", len(subs))
print("chaves de substitution:", Counter(k for _, e in subs for k in e["substitution"]))
print("replacement tem id e name:", all({"id", "name"} <= set(e["substitution"]["replacement"]) for _, e in subs))
print("substitution.outcome:"); show(Counter(e["substitution"].get("outcome", {}).get("name", "<ausente>") for _, e in subs))
print("evento tem 'player' (quem sai):", all("player" in e for _, e in subs))

# Conferência: quem entra aparece na escalação com uma posição iniciada por substituição?
entered = {(mid, pl["player_id"]) for mid, lu in lineups.items() for t in lu for pl in t["lineup"]
           if pl["positions"] and pl["positions"][0]["start_reason"].startswith("Substitution")}
print("jogadores que entram, encontrados na escalação como reservas utilizados:",
      sum((mid, e["substitution"]["replacement"]["id"]) in entered for mid, e in subs), "de", len(subs))

substituições: 587
chaves de substitution: Counter({'outcome': 587, 'replacement': 587})
replacement tem id e name: True
substitution.outcome:
     547  Tactical
      40  Injury
evento tem 'player' (quem sai): True
jogadores que entram, encontrados na escalação como reservas utilizados: 531 de 587


## 10. `Half End` e duração dos períodos

A M8 precisa da duração real de cada período. Perguntas:
- Quantos `Half End` existem por período?
- Os dois times registram o mesmo horário?
- Qual partida teve o 1º tempo mais longo?

In [16]:
half_end = defaultdict(list)
for mid, e in all_events:
    if e["type"]["name"] == "Half End":
        half_end[(mid, e["period"])].append(timestamp_to_seconds(e["timestamp"]))
print("Half End por (partida, período):", Counter(len(v) for v in half_end.values()))
print("pares com horários diferentes entre os times:", sum(len(set(v)) > 1 for v in half_end.values()),
      "| maior diferença (s):", round(max(max(v) - min(v) for v in half_end.values()), 3))

first_half = sorted(((max(v), mid) for (mid, p), v in half_end.items() if p == 1), reverse=True)
name = {m["match_id"]: f'{m["home_team"]["home_team_name"]} x {m["away_team"]["away_team_name"]}' for m in matches}
print("\n1º tempo mais longo (duração real, mm:ss):")
for secs, mid in first_half[:3]:
    print(f"  {mid}  {int(secs // 60)}:{int(secs % 60):02d}  {name[mid]}")

Half End por (partida, período): Counter({2: 143})
pares com horários diferentes entre os times: 0 | maior diferença (s): 0.0

1º tempo mais longo (duração real, mm:ss):
  3857271  59:03  England x Iran
  3857297  55:02  Poland x Saudi Arabia
  3857293  53:07  Ghana x Uruguay


## 11. Integridade entre arquivos

Estas consultas viram checks do Passo 9:
- **C02:** placar reconstruído = placar oficial.
- **C08:** o time de cada evento é mandante ou visitante.
- **FK:** todo jogador dos eventos está nas escalações.
- **C11:** um mesmo jogador tem sempre o mesmo nome.

In [17]:
home_away = {m["match_id"]: (m["home_team"]["home_team_id"], m["away_team"]["away_team_id"]) for m in matches}

divergences = []
for m in matches:
    goals = Counter()
    for e in events[m["match_id"]]:
        if e["period"] >= 5:
            continue
        if e["type"]["name"] == "Shot" and e["shot"]["outcome"]["name"] == "Goal":
            goals[e["team"]["id"]] += 1
        if e["type"]["name"] == "Own Goal For":
            goals[e["team"]["id"]] += 1
    h, a = home_away[m["match_id"]]
    if (goals[h], goals[a]) != (m["home_score"], m["away_score"]):
        divergences.append(m["match_id"])
print("C02 — partidas com placar divergente:", divergences or "nenhuma")

print("C08 — eventos de time fora da partida:",
      sum(e["team"]["id"] not in home_away[mid] for mid, e in all_events),
      "| posse de time fora da partida:", sum(e["possession_team"]["id"] not in home_away[mid] for mid, e in all_events))

lineup_players = {(mid, pl["player_id"]) for mid, lu in lineups.items() for t in lu for pl in t["lineup"]}
print("FK — jogadores dos eventos ausentes da escalação:",
      sum("player" in e and (mid, e["player"]["id"]) not in lineup_players for mid, e in all_events),
      "| recebedores de passe ausentes:",
      sum("recipient" in e["pass"] and (mid, e["pass"]["recipient"]["id"]) not in lineup_players for mid, e in passes))

names = defaultdict(set)
for lu in lineups.values():
    for t in lu:
        for pl in t["lineup"]:
            names[pl["player_id"]].add(pl["player_name"])
print("C11 — jogadores distintos:", len(names), "| com mais de um nome:", sum(len(v) > 1 for v in names.values()))

name_freq = defaultdict(Counter)
for lu in lineups.values():
    for t in lu:
        for pl in t["lineup"]:
            name_freq[pl["player_id"]][pl["player_name"]] += 1
for pid, freq in name_freq.items():
    if len(freq) > 1:
        print(f"  player_id {pid}: {dict(freq)}")

C02 — partidas com placar divergente: nenhuma
C08 — eventos de time fora da partida: 0 | posse de time fora da partida: 0
FK — jogadores dos eventos ausentes da escalação: 0 | recebedores de passe ausentes: 0
C11 — jogadores distintos: 829 | com mais de um nome: 4
  player_id 4354: {'Phil Foden': 3, 'Philip Foden': 2}
  player_id 50494: {'Yeremy Jesús Pino Santos': 3, 'Yeremi Jesús Pino Santos': 1}
  player_id 10603: {"Mamadou Loum N'Diaye": 3, "Mamadou Loum N''Diaye": 1}
  player_id 24475: {'Awer Mabil': 1, 'Awer Denis Bul Mabil': 3}


## 12. Partidas-sentinela

Três partidas pequenas, escolhidas **pelos dados**, cobrem os casos difíceis antes de escalar para as 64 partidas:
1. **A final:** disputa de pênaltis, prorrogação e pênaltis durante o jogo.
2. **Uma partida com gol contra:** exercita a M1 e a validação do placar.
3. **A partida com o 1º tempo mais longo:** exercita a ordenação da M8.

In [18]:
date = {m["match_id"]: m["match_date"] for m in matches}
own_goals = sorted({mid for mid, e in all_events if e["type"]["name"] == "Own Goal For"}, key=lambda x: (date[x], x))
print("partidas com gol contra:")
for mid in own_goals:
    print(f"  {mid}  {date[mid]}  {name[mid]}")
print("partidas com disputa de pênaltis:", sorted({mid for mid, e in all_events if e["period"] == 5}))
print("partida com o 1º tempo mais longo:", first_half[0][1], name[first_half[0][1]])

partidas com gol contra:
  3857276  2022-12-01  Canada x Morocco
  3857292  2022-12-01  Costa Rica x Germany
  3869151  2022-12-03  Argentina x Australia
partidas com disputa de pênaltis: [3869219, 3869220, 3869321, 3869420, 3869685]
partida com o 1º tempo mais longo: 3857271 England x Iran


## 13. Valores de referência da final (M1–M8)

As regras estão implementadas **exatamente** como em `docs/metrics.md`, em Python puro. Estes números viram `tests/fixtures/reference_final_3869685.json`, **copiados à mão** depois de revisados. As views SQL do Passo 10 precisam reproduzi-los.

In [19]:
OPEN_PLAY = {None, "Recovery", "Interception"}
ON_TARGET = {"Goal", "Saved", "Saved to Post"}
MIN_START_X, MIN_GAIN_YD = 48, 10


def in_box(x, y):
    return x >= 102 and 18 <= y <= 62


def goal_distance(x, y):
    return hypot(120 - x, 40 - y)


def is_progressive(x0, y0, x1, y1):
    enters_box = in_box(x1, y1) and not in_box(x0, y0)
    return x0 >= MIN_START_X and (goal_distance(x0, y0) - goal_distance(x1, y1) >= MIN_GAIN_YD or enters_box)


ev = events[FINAL_ID]
team_name = {e["team"]["id"]: e["team"]["name"] for e in ev}
ref = {tid: Counter() for tid in team_name}
player_goals = Counter()
steps = defaultdict(list)

for e in ev:
    t, kind = e["team"]["id"], e["type"]["name"]
    r = ref[t]
    if e["period"] == 5:
        if kind == "Shot" and e["shot"]["outcome"]["name"] == "Goal":
            r["penaltis_disputa"] += 1
        continue
    if kind == "Own Goal For":
        r["gols"] += 1
        r["gols_contra_a_favor"] += 1
    if kind == "Shot":
        s = e["shot"]
        xg = s["statsbomb_xg"]
        r["finalizacoes"] += 1
        r["finalizacoes_no_alvo"] += s["outcome"]["name"] in ON_TARGET
        r["xg"] += xg
        if s["outcome"]["name"] == "Goal":
            r["gols"] += 1
            player_goals[e["player"]["name"]] += 1
        if s["type"]["name"] != "Penalty":
            r["npxg"] += xg
            r["finalizacoes_sem_penalti"] += 1
        steps[t].append((e["minute"], s["type"]["name"], xg))
    if kind == "Pass":
        p = e["pass"]
        ptype = p.get("type", {}).get("name")
        outcome = p.get("outcome", {}).get("name")
        complete = outcome is None
        if ptype != "Throw-in" and outcome not in ("Unknown", "Injury Clearance"):
            r["passes_tentados"] += 1
            r["passes_completos"] += complete
        if complete and ptype in OPEN_PLAY:
            (x0, y0), (x1, y1) = e["location"][:2], p["end_location"][:2]
            r["passes_progressivos"] += is_progressive(x0, y0, x1, y1)
            r["passes_para_area"] += in_box(x1, y1) and not in_box(x0, y0)
    if kind == "Carry":
        (x0, y0), (x1, y1) = e["location"][:2], e["carry"]["end_location"][:2]
        r["conducoes_progressivas"] += is_progressive(x0, y0, x1, y1)

reference = {}
for tid, r in ref.items():
    reference[team_name[tid]] = {
        "team_id": tid,
        "gols": r["gols"],
        "gols_contra_a_favor": r["gols_contra_a_favor"],
        "penaltis_disputa": r["penaltis_disputa"],
        "finalizacoes": r["finalizacoes"],
        "finalizacoes_no_alvo": r["finalizacoes_no_alvo"],
        "xg": round(r["xg"], 4),
        "npxg": round(r["npxg"], 4),
        "finalizacoes_sem_penalti": r["finalizacoes_sem_penalti"],
        "npxg_por_finalizacao_sem_penalti": round(r["npxg"] / r["finalizacoes_sem_penalti"], 4),
        "passes_tentados": r["passes_tentados"],
        "passes_completos": r["passes_completos"],
        "pct_acerto_passes": round(100 * r["passes_completos"] / r["passes_tentados"], 1),
        "passes_progressivos": r["passes_progressivos"],
        "conducoes_progressivas": r["conducoes_progressivas"],
        "passes_para_area": r["passes_para_area"],
        "xg_acumulado_final": round(sum(x for _, _, x in steps[tid]), 4),
        "degraus_de_penalti_campo_minute": [m for m, kind, _ in steps[tid] if kind == "Penalty"],
    }
print(json.dumps(reference, indent=1, ensure_ascii=False))
print("gols por jogador:", dict(player_goals))
print("\nobs.: 'minute' é o minuto decorrido (começa em 0). No rótulo de futebol soma-se 1: minute 22 -> 23'.")

{
 "Argentina": {
  "team_id": 779,
  "gols": 3,
  "gols_contra_a_favor": 0,
  "penaltis_disputa": 4,
  "finalizacoes": 20,
  "finalizacoes_no_alvo": 10,
  "xg": 2.7583,
  "npxg": 1.9748,
  "finalizacoes_sem_penalti": 19,
  "npxg_por_finalizacao_sem_penalti": 0.1039,
  "passes_tentados": 661,
  "passes_completos": 538,
  "pct_acerto_passes": 81.4,
  "passes_progressivos": 56,
  "conducoes_progressivas": 30,
  "passes_para_area": 7,
  "xg_acumulado_final": 2.7583,
  "degraus_de_penalti_campo_minute": [
   22
  ]
 },
 "France": {
  "team_id": 771,
  "gols": 3,
  "gols_contra_a_favor": 0,
  "penaltis_disputa": 2,
  "finalizacoes": 10,
  "finalizacoes_no_alvo": 5,
  "xg": 2.2726,
  "npxg": 0.7056,
  "finalizacoes_sem_penalti": 8,
  "npxg_por_finalizacao_sem_penalti": 0.0882,
  "passes_tentados": 544,
  "passes_completos": 414,
  "pct_acerto_passes": 76.1,
  "passes_progressivos": 39,
  "conducoes_progressivas": 31,
  "passes_para_area": 3,
  "xg_acumulado_final": 2.2726,
  "degraus_de_pena

## Conclusões

As conclusões desta exploração (commit fixado, os 7 caminhos ⚠️ resolvidos, as sentinelas e as diferenças encontradas) estão registradas em [`docs/data_notes.md`](../docs/data_notes.md).